
# Diebold--Mariano tests for pricing and hedging results

This notebook produces the two Diebold--Mariano test tables reported in the thesis:

1. pairwise out-of-sample pricing comparisons between Black, Heston, and SVCJ at $\lambda_{TS}=100$, separately for BTC and ETH;
2. regular-delta versus premium-adjusted net-delta hedging comparisons under the representative Black model, separately for BTC and ETH.


In [ ]:
from pathlib import Path
import math
import pandas as pd
from scipy.stats import norm


## Configuration and helper functions

For each model pair $(a,b)$, the loss differential is defined as

$$
d_t^{a,b} = L_{a,t} - L_{b,t}.
$$

Thus, a positive mean loss differential means that model or hedge rule $b$ has lower loss than $a$.

The Diebold--Mariano statistic is computed as

$$
DM = \frac{\bar d}{\widehat{\mathrm{se}}_{NW}(\bar d)},
$$

where the Newey--West estimator uses 8 lags.

In [ ]:
NW_LAGS = 8
CALIBRATION_FILE = "calibration_results_reg_100.xlsx"
HEDGING_FILE = "hedging_results.xlsx"


def find_repo_root_with_excel_dir() -> Path:
    """Find a parent directory containing the expected 'excel files' folder."""
    start = Path.cwd().resolve()
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        excel_dir = candidate / "excel files"
        if (excel_dir / CALIBRATION_FILE).exists() and (excel_dir / HEDGING_FILE).exists():
            return candidate
    checked = "\n".join(str(c / "excel files") for c in candidates)
    raise FileNotFoundError(
        "Could not find an 'excel files' folder containing both required workbooks. "
        f"Checked:\n{checked}"
    )


def normal_two_sided_pvalue(z: float) -> float:
    """Two-sided p-value for a standard normal statistic."""
    return 2.0 * norm.sf(abs(z))


def newey_west_dm_test(values, lags: int = NW_LAGS) -> dict:
    """DM test for a single loss differential series."""
    d = [float(x) for x in values if pd.notna(x) and math.isfinite(float(x))]
    n = len(d)
    if n < 2:
        raise ValueError("At least two valid observations are required.")

    mean = sum(d) / n
    centered = [x - mean for x in d]

    # Newey--West long-run variance of sqrt(n) * mean(d_t).
    # Autocovariances are divided by n, matching the implementation used for the thesis tables.
    gamma0 = sum(x * x for x in centered) / n
    lrv = gamma0
    max_lag = min(lags, n - 1)
    for ell in range(1, max_lag + 1):
        gamma = sum(centered[i] * centered[i - ell] for i in range(ell, n)) / n
        weight = 1.0 - ell / (lags + 1.0)
        lrv += 2.0 * weight * gamma

    se = math.sqrt(lrv / n)
    dm_stat = mean / se
    p_value = normal_two_sided_pvalue(dm_stat)
    ci_low = mean - 1.96 * se
    ci_high = mean + 1.96 * se

    return {
        "n": n,
        "mean": mean,
        "se": se,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "dm_stat": dm_stat,
        "p_value": p_value,
        "long_run_variance": lrv,
    }


def fmt_num(x: float, digits: int = 3) -> str:
    return f"{x:.{digits}f}"


def fmt_pvalue(x: float) -> str:
    return f"{x:.2e}"


repo_root = find_repo_root_with_excel_dir()
excel_dir = repo_root / "excel files"
calibration_path = excel_dir / CALIBRATION_FILE
hedging_path = excel_dir / HEDGING_FILE

print(f"Repository root: {repo_root}")
print(f"Calibration workbook: {calibration_path}")
print(f"Hedging workbook: {hedging_path}")


Repository root: /mnt/data/dm_notebook_test_repo
Calibration workbook: /mnt/data/dm_notebook_test_repo/excel files/calibration_results_reg_100.xlsx
Hedging workbook: /mnt/data/dm_notebook_test_repo/excel files/hedging_results.xlsx



## 1. Pricing DM tests

The pricing loss for model $m$ at snapshot $t$ is the out-of-sample mean squared pricing error:

$$
L_{m,t} = \mathrm{RMSE}_{m,t}^2.
$$

The workbook already contains the snapshot-level test RMSE in the model parameter sheets, so the loss differential for a model pair $(a,b)$ is computed as

$$
d_t^{a,b} = \mathrm{RMSE}_{a,t}^2 - \mathrm{RMSE}_{b,t}^2.
$$


In [3]:

def load_pricing_losses(calibration_workbook: Path) -> pd.DataFrame:
    sheets = {
        "black": "black_params",
        "heston": "heston_params",
        "svcj": "svcj_params",
    }
    merged = None
    for model, sheet_name in sheets.items():
        df = pd.read_excel(calibration_workbook, sheet_name=sheet_name)
        df = df[["timestamp", "currency", "rmse_test"]].copy()
        df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
        df = df.rename(columns={"rmse_test": f"rmse_{model}"})
        merged = df if merged is None else merged.merge(df, on=["timestamp", "currency"], how="outer")
    merged = merged.sort_values(["currency", "timestamp"]).reset_index(drop=True)
    for model in sheets:
        merged[f"loss_{model}"] = merged[f"rmse_{model}"] ** 2
    return merged


def pricing_dm_table(pricing_losses: pd.DataFrame) -> pd.DataFrame:
    model_pairs = [
        ("black", "heston", "Black, Heston"),
        ("heston", "svcj", "Heston, SVCJ"),
        ("black", "svcj", "Black, SVCJ"),
    ]
    rows = []
    for currency in ["BTC", "ETH"]:
        sub = pricing_losses.loc[pricing_losses["currency"] == currency].copy()
        for model_a, model_b, label in model_pairs:
            d = sub[f"loss_{model_a}"] - sub[f"loss_{model_b}"]
            result = newey_west_dm_test(d, lags=NW_LAGS)
            rows.append({
                "Currency": currency,
                "Model pair (a,b)": label,
                "Snapshots": result["n"],
                "Mean d x 1e6": result["mean"] * 1e6,
                "95% CI x 1e6": f"[{result['ci_low'] * 1e6:.3f}, {result['ci_high'] * 1e6:.3f}]".replace("[ ", "["),
                "DM statistic": result["dm_stat"],
                "p-value": result["p_value"],
            })
    return pd.DataFrame(rows)


pricing_losses = load_pricing_losses(calibration_path)
pricing_results = pricing_dm_table(pricing_losses)

pricing_table = pricing_results.copy()
pricing_table["Mean d x 1e6"] = pricing_table["Mean d x 1e6"].map(lambda x: fmt_num(x, 3))
pricing_table["DM statistic"] = pricing_table["DM statistic"].map(lambda x: fmt_num(x, 2))
pricing_table["p-value"] = pricing_table["p-value"].map(fmt_pvalue)
pricing_table


,Currency,"Model pair (a,b)",Snapshots,Mean d x 1e6,95% CI x 1e6,DM statistic,p-value
0,BTC,"Black, Heston",388,21.287,"[17.509, 25.064]",11.05,2.30e-28
1,BTC,"Heston, SVCJ",387,1.715,"[1.391, 2.039]",10.38,3.04e-25
2,BTC,"Black, SVCJ",387,22.442,"[19.095, 25.788]",13.14,1.85e-39
3,ETH,"Black, Heston",388,18.312,"[12.623, 24.000]",6.31,2.80e-10
4,ETH,"Heston, SVCJ",388,2.189,"[1.521, 2.858]",6.42,1.39e-10
5,ETH,"Black, SVCJ",388,20.501,"[15.145, 25.857]",7.50,6.28e-14



## 2. Hedging DM tests

For the representative Black model, the snapshot-level hedging loss for hedge rule $h$ is

$$
L_{h,t}^{Black}
= \frac{1}{N_t}\sum_{i=1}^{N_t}\left(\varepsilon_{t,i}^{Black,h}\right)^2.
$$

The `hedge_summary_by_timestamp` sheet already reports the corresponding snapshot-level hedged RMSE. Hence the loss differential is

$$
d_t^{\Delta,net}
= \mathrm{RMSE}_{\Delta,t}^2 - \mathrm{RMSE}_{\Delta^{net},t}^2.
$$

Positive values imply lower mean squared hedging error for the premium-adjusted net-delta hedge.


In [4]:

def load_black_hedging_losses(hedging_workbook: Path) -> pd.DataFrame:
    df = pd.read_excel(hedging_workbook, sheet_name="hedge_summary_by_timestamp")
    df = df[
        (df["split"] == "test")
        & (df["model"] == "black")
        & (df["hedge_type"].isin(["delta", "net_delta"]))
    ].copy()
    df["snapshot_ts"] = pd.to_datetime(df["snapshot_ts"], utc=True, errors="coerce")
    pivot = (
        df.pivot_table(
            index=["snapshot_ts", "currency"],
            columns="hedge_type",
            values="rmse_hedged_coin",
            aggfunc="first",
        )
        .reset_index()
        .sort_values(["currency", "snapshot_ts"])
        .reset_index(drop=True)
    )
    pivot["loss_delta"] = pivot["delta"] ** 2
    pivot["loss_net_delta"] = pivot["net_delta"] ** 2
    return pivot


def hedging_dm_table(hedging_losses: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for currency in ["BTC", "ETH"]:
        sub = hedging_losses.loc[hedging_losses["currency"] == currency].copy()
        d = sub["loss_delta"] - sub["loss_net_delta"]
        result = newey_west_dm_test(d, lags=NW_LAGS)
        rows.append({
            "Currency": currency,
            "Snapshots": result["n"],
            "Mean d x 1e6": result["mean"] * 1e6,
            "95% CI x 1e6": f"[{result['ci_low'] * 1e6:.3f}, {result['ci_high'] * 1e6:.3f}]".replace("[ ", "["),
            "DM statistic": result["dm_stat"],
            "p-value": result["p_value"],
        })
    return pd.DataFrame(rows)


hedging_losses = load_black_hedging_losses(hedging_path)
hedging_results = hedging_dm_table(hedging_losses)

hedging_table = hedging_results.copy()
hedging_table["Mean d x 1e6"] = hedging_table["Mean d x 1e6"].map(lambda x: fmt_num(x, 3))
hedging_table["DM statistic"] = hedging_table["DM statistic"].map(lambda x: fmt_num(x, 2))
hedging_table["p-value"] = hedging_table["p-value"].map(fmt_pvalue)
hedging_table


,Currency,Snapshots,Mean d x 1e6,95% CI x 1e6,DM statistic,p-value
0,BTC,387,8.858,"[5.503, 12.214]",5.17,2.29e-07
1,ETH,387,24.117,"[16.920, 31.314]",6.57,5.10e-11
